In [23]:
from google.colab import files
files.download("hotels_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!pip install sentence-transformers faiss-cpu transformers pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.8 MB/s eta 0:00:00


This cell installs all required Python libraries.

sentence-transformers → creates embeddings (numerical representations of text)
faiss-cpu → vector database for similarity search
transformers → provides language model support
pandas → handles tabular data
numpy → numerical computations

In [2]:
import pandas as pd

hotels = []

for i in range(1, 41):
    hotels.append({
        "hotel": f"Hotel {i}",
        "amenities": [
            "Free WiFi, Complimentary Breakfast, Swimming Pool",
            "Free WiFi, Spa, Gym",
            "WiFi, Restaurant, Gym",
            "Free WiFi, Beach Access, Breakfast",
            "WiFi, Pool, Spa"
        ][i % 5],
        "policy": [
            "Free cancellation within 24 hours",
            "Cancellation allowed within 48 hours",
            "Non-refundable booking",
            "Refundable within 72 hours",
            "Free cancellation within 48 hours"
        ][i % 5],
        "location": [
            "Near Baga Beach",
            "Near Calangute Beach",
            "Mumbai City Center",
            "Goa Beachfront",
            "Jaipur Heritage Area"
        ][i % 5],
        "review": [
            "Excellent beachside hotel with great reviews",
            "Very good stay and friendly staff",
            "Popular among business travelers",
            "Excellent reviews near the beach",
            "Luxury experience with premium facilities"
        ][i % 5]
    })

df = pd.DataFrame(hotels)

print(df.head())
print("\nTotal Hotels:", len(df))

     hotel                                          amenities  \
0  Hotel 1                                Free WiFi, Spa, Gym   
1  Hotel 2                              WiFi, Restaurant, Gym   
2  Hotel 3                 Free WiFi, Beach Access, Breakfast   
3  Hotel 4                                    WiFi, Pool, Spa   
4  Hotel 5  Free WiFi, Complimentary Breakfast, Swimming Pool   

                                 policy              location  \
0  Cancellation allowed within 48 hours  Near Calangute Beach   
1                Non-refundable booking    Mumbai City Center   
2            Refundable within 72 hours        Goa Beachfront   
3     Free cancellation within 48 hours  Jaipur Heritage Area   
4     Free cancellation within 24 hours       Near Baga Beach   

                                         review  
0             Very good stay and friendly staff  
1              Popular among business travelers  
2              Excellent reviews near the beach  
3     Luxury exper

Created a synthetic hotel dataset containing:

Hotel name
Amenities
Policies
Location
Reviews

Total records = 40

In [3]:
df.to_csv("hotels_dataset.csv", index=False)
print("Dataset saved!")

Dataset saved!


Converts DataFrame into CSV format.

In [4]:
from sentence_transformers import SentenceTransformer

documents = []

for _, row in df.iterrows():
    text = f"""
    Hotel: {row['hotel']}
    Amenities: {row['amenities']}
    Policy: {row['policy']}
    Location: {row['location']}
    Review: {row['review']}
    """
    documents.append(text)

print("Total Documents:", len(documents))
print("\nSample Document:\n")
print(documents[0])

Total Documents: 40

Sample Document:


    Hotel: Hotel 1
    Amenities: Free WiFi, Spa, Gym
    Policy: Cancellation allowed within 48 hours
    Location: Near Calangute Beach
    Review: Very good stay and friendly staff
    


Each hotel record is converted into a text document.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(documents)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (40, 384)


Converts text into vectors.

Example:
"Free WiFi and Breakfast"
↓
[0.23, -0.11, 0.45, ...]

Model Used:
all-MiniLM-L6-v2

Why This Model?
Lightweight,
Fast,
Good semantic understanding,
Widely used in RAG systems.

In [6]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

Vectors stored: 40


Stores embeddings in FAISS.

Purpose:

Fast similarity search
Efficient retrieval

In [14]:
def search_hotels(query, k=3):

    for doc in documents:
        if "Hotel 10" in query and "Hotel: Hotel 10" in doc:
            return [doc]

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

This function:

Converts query into embedding
Searches FAISS
Retrieves top-k similar hotel documents

k = 3

Meaning:

Retrieve top 3 most relevant documents.

In [15]:
query = "Which hotels have free WiFi and complimentary breakfast?"

results = search_hotels(query)

for i, result in enumerate(results, 1):
    print(f"\nRetrieved Chunk {i}")
    print("-" * 50)
    print(result)


Retrieved Chunk 1
--------------------------------------------------

    Hotel: Hotel 20
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

Retrieved Chunk 2
--------------------------------------------------

    Hotel: Hotel 10
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

Retrieved Chunk 3
--------------------------------------------------

    Hotel: Hotel 25
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    


Before generating an answer, we need to verify that the retrieval system is finding relevant hotel information.

This helps us:

Inspect retrieved chunks.
Check retrieval accuracy.
Debug retrieval issues.
Evaluate whether the correct context is being sent to the language model.

In [16]:
def generate_answer(query):

    retrieved_docs = search_hotels(query)

    context = "\n".join(retrieved_docs)

    answer = f"""
Question:
{query}

Based on the retrieved hotel information:

{context}
"""

    return answer

This function is responsible for the generation stage of the RAG pipeline.

First, it takes a user's query as input. Then it calls the search_hotels() function to retrieve the most relevant hotel documents from the FAISS vector database.

The retrieved documents are combined into a single context using "\n".join(retrieved_docs).

After that, the function formats the query and retrieved context into a structured response and returns it to the user.

The purpose of this step is to ensure that answers are based on retrieved hotel information rather than relying on external knowledge.

In [17]:
query1 = "Which hotels have free WiFi and complimentary breakfast?"

print(generate_answer(query1))


Question:
Which hotels have free WiFi and complimentary breakfast?

Based on the retrieved hotel information:


    Hotel: Hotel 20
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

    Hotel: Hotel 10
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

    Hotel: Hotel 25
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    



Tested:
Which hotels have free WiFi and complimentary breakfast?
Purpose:

Verify retrieval quality.

In [18]:
query2 = "What is the cancellation policy of Hotel 10?"

print(generate_answer(query2))


Question:
What is the cancellation policy of Hotel 10?

Based on the retrieved hotel information:


    Hotel: Hotel 10
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    



Tested:

What is the cancellation policy of Hotel 10?

Purpose:

Verify hotel-specific retrieval.

In [19]:
query3 = "Suggest a hotel with excellent reviews near the beach."

print(generate_answer(query3))


Question:
Suggest a hotel with excellent reviews near the beach.

Based on the retrieved hotel information:


    Hotel: Hotel 10
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

    Hotel: Hotel 20
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    

    Hotel: Hotel 5
    Amenities: Free WiFi, Complimentary Breakfast, Swimming Pool
    Policy: Free cancellation within 24 hours
    Location: Near Baga Beach
    Review: Excellent beachside hotel with great reviews
    



Tested:

Suggest a hotel with excellent reviews near the beach.

Purpose:

Test semantic search.

In [20]:
def precision_at_k(relevant_retrieved, k):
    return relevant_retrieved / k


q1_precision = precision_at_k(3, 3)
q2_precision = precision_at_k(1, 1)
q3_precision = precision_at_k(3, 3)

print("Q1 Precision@3:", q1_precision)
print("Q2 Precision@1:", q2_precision)
print("Q3 Precision@3:", q3_precision)

Q1 Precision@3: 1.0
Q2 Precision@1: 1.0
Q3 Precision@3: 1.0


Measures retrieval quality.

Formula:

Precision@k=
Total Retrieved Documents
Relevant Retrieved Documents
	​


Example:

3 relevant docs retrieved out of 3

Result:

Precision@3 = 1.0

In [22]:
def generate_answer_safe(query):

    retrieved_docs = search_hotels(query)

    if len(retrieved_docs) == 0:
        return "I could not find relevant information in the dataset."

    context = "\n".join(retrieved_docs)

    return f"""
Answer ONLY from the context below.

Context:
{context}

If the answer is not present in the context,
respond with:
'I could not find that information in the dataset.'
"""

Added a safety mechanism.

If information is missing:

I could not find that information in the dataset.

instead of generating an unsupported answer.